# 04 · Price Forecasting
Predict `unit_price` from component, sourcing, contract, order and **real macro** features. Compare a linear baseline against Random Forest and Gradient Boosting.

In [ ]:
%matplotlib inline
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40, "display.width", 160)
from src import config as C
print("project root:", ROOT)

In [ ]:
from src import price_model as pm
df = pd.read_csv(C.PROCESSED_QUOTES_CSV)
art = pm.train_and_evaluate(df)
art.metrics

Non-linear models beat the linear baseline, especially on **MAPE**, because the price drivers act multiplicatively.

In [ ]:
yt = art.y_test; yp = art.predictions[art.best_name]
fig, axes = plt.subplots(1,2, figsize=(14,6))
axes[0].scatter(yt, yp, alpha=.25, s=18, color='#1f77b4')
lim=[min(yt.min(),yp.min()), max(yt.max(),yp.max())]
axes[0].plot(lim,lim,'--',color='#b3122b'); axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_title(f'Actual vs predicted — {art.best_name}'); axes[0].set_xlabel('actual'); axes[0].set_ylabel('predicted')
res = yp - yt
axes[1].hist(res, bins=50, color='#4c72b0'); axes[1].axvline(0,color='#b3122b')
axes[1].set_title('Residuals (pred - actual)'); plt.tight_layout(); plt.show()

## Feature importance

In [ ]:
fi = art.feature_importance.head(12).iloc[::-1]
labels = fi.feature.str.replace('cat__','',regex=False).str.replace('num__','',regex=False).str.replace('_',' ')
fig, ax = plt.subplots(figsize=(10,6)); ax.barh(labels, fi.importance, color='#4c72b0')
ax.set_title(f'Feature importance — {art.best_name}'); plt.show()

**Read-out.** Component identity sets the price tier; logistics cost, energy index, lead time, order volume and the commodity index drive variation within a tier. The model gives buyers an *expected price* to flag quotes that sit materially above it.